In [3]:
from dotenv import load_dotenv
from src.db.chroma_db import ChromaDb
from src.models.openai_provider import OpenAILLMProvider, OpenAIEmbeddingProvider
from src.prompts import timeframe_detection_system, timeframe_detection_user
from src.schemas.schemas import TimeframeDetection, InputLanguage
from src.services.nkod_data_processor import NkodDataProcessor
from src.db.graph_db import GraphDb
from src.db.sq_lite import SqLite
from datetime import date
from src.services.language_detector import LanguageDetector
from src.services.nkod_query_matcher import NkodQueryMatcher
from src.services.timeframe_detector import TimeframeDetector
from src.evaluators.nkod_query_matcher_evaluator import NkodQueryMatcherEvaluator
from src.models.google_provider import GeminiEmbeddingProvider
from src.services.nkod_query_matcher_reranker import NkodQueryMatcherReranker
from src.services.nkod_rag import NkodRAG


load_dotenv()

True

## Downloading and creating SQL tables for the NKOD metadata

In [2]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)
sq_lite = SqLite(nkod_data_processor.metadata_sql_path)

#nkod_data_processor.create_dataset_publisher_csv(graph_db)
#nkod_data_processor.download_catalog_metadata()
#nkod_data_processor.create_metadata_csv(graph_db)
#nkod_data_processor.create_themes_csv(graph_db)
#nkod_data_processor.create_metadata_sql(sq_lite)
#nkod_data_processor.create_themes_sql(sq_lite)

## Indexing the keywords, titles and descriptions from the NKOD metadata

In [2]:
openai_embeddings = OpenAIEmbeddingProvider(model_name="text-embedding-3-large", dimensions=1536)
chroma_db = ChromaDb(nkod_data_processor.vectordb_path)

#nkod_data_processor.index_catalog_themes(sq_lite, openai_embeddings, chroma_db)
#nkod_data_processor.index_catalog_metadata(sq_lite, openai_embeddings, chroma_db, verbose=True)
print(chroma_db.list_collections())

NameError: name 'OpenAIEmbeddingProvider' is not defined

## Language detection, Timeframe detection and Query matching on OFN dataset

In [4]:
model_name ="gpt-5"
openai_llm = OpenAILLMProvider(
    model_name=model_name,
    temperature=1.0
)
nkod_query_evaluator = NkodQueryMatcherEvaluator()
nkod_query_reranker = NkodQueryMatcherReranker()

k = 30
evaluation_dataset = "ofn_dataset_ofn_new.jsonl"
nkod_query_evaluator.evaluate_on_ofn_dataset(k, evaluation_dataset, chroma_db, nkod_data_processor, "cs", openai_embeddings, nkod_query_reranker, openai_llm)

Query 1/11
Desc: jednoduchý dotaz
Original query: Jaké jsou aktuality v Říčanech?
Cleaned query: jaké jsou aktuality v říčanech?
In titles: True -> 1/1 present | positions: [0]
In titles rerankedTrue -> 1/1 present | positions: [0]
Titles similarity matching: [(0.3168889284133911, 'aktuality města říčany'), (0.3986581563949585, 'aktuální stav na povodňových čidlech ústeckého kraje'), (0.41426151990890503, 'události ve městě říčany'), (0.42163896560668945, 'aktuality'), (0.42163896560668945, 'aktuality'), (0.42198145389556885, 'georizika - vodní toky'), (0.4572067856788635, 'vodní toky'), (0.46155452728271484, 'množství povrchových vod – údaje – recent'), (0.4615558385848999, 'množství povrchových vod – údaje – recent'), (0.470719575881958, 'pětisetletá voda q500 - záplavové území - liberecký kraj'), (0.47981637716293335, 'dvacetiletá voda q20 - záplavové území - liberecký kraj'), (0.48033618927001953, 'akumulace vod ve vodních nádržích 2021'), (0.48201149702072144, 'riparian zones - bř

## Language detection, Timeframe detection and Query matching on LLM dataset

In [5]:
# TODO: doplnit evaluaci na LLM datasetu

In [5]:
from langchain.chains import GraphSparqlQAChain
from langchain_community.graphs import RdfGraph
from langchain_openai import ChatOpenAI
from rdflib import Graph

# https://www.ontotext.com/blog/natural-language-querying-of-graphdb-in-langchain/
#print(RdfGraph(source_file="https://vyzlovka.cz/4w-opendata.php?agenda=turisticke-cile", serialization="json-ld").schema)
chain = GraphSparqlQAChain.from_llm(
    ChatOpenAI(model="gpt-5", temperature=1), graph=RdfGraph(source_file="https://www.horsovskytyn.cz/api/open-data/ofn63db8a485d463", serialization="json-ld"), verbose=True, allow_dangerous_requests=True,
)

chain.invoke("aktualita ohledne ridicaku")




> Entering new GraphSparqlQAChain chain...
Identified intent:
SELECT
Generated SPARQL:
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX schema: <http://schema.org/>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT DISTINCT ?article ?title ?description ?modified
WHERE {
  ?article rdf:type schema:NewsArticle .
  OPTIONAL { ?article dcterms:title ?title . }
  OPTIONAL { ?article dcterms:description ?description . }
  OPTIONAL { ?article dcterms:modified ?modified . }
  {
    { ?article dcterms:title ?t . }
    UNION
    { ?article dcterms:description ?t . }
  }
  FILTER regex(str(?t), "(?i)(ridicak|řidičák|řidič)")
}
ORDER BY DESC(?modified)
Full Context:
[(rdflib.term.URIRef('https://www.horsovskytyn.cz/ofn/zprávy/ofn63db8a485d463/ceka-vas-vymena-ridicskeho-prukazu-1313cs'), rdflib.term.Literal('Čeká vás výměna řidičského průkazu?', lang='cs'), rdflib.term.Literal('Čeká vás výměna řidičského průkazu?\r\nNa úřad už nemusíte. \r\nPožádat o nový si můžete prostřednictv

{'query': 'aktualita ohledne ridicaku',
 'result': 'Jako AI asistent mám k dispozici tuto aktuální informaci k řidičským průkazům:\n\nNázev: Čeká vás výměna řidičského průkazu?\nPopis: Čeká vás výměna řidičského průkazu? Na úřad už nemusíte. Požádat o nový si můžete prostřednictvím Portálu dopravy.\nOdkaz: https://www.horsovskytyn.cz/ofn/zprávy/ofn63db8a485d463/ceka-vas-vymena-ridicskeho-prukazu-1313cs'}

In [ ]:
from rdflib import Graph
g = Graph()
g.parse("https://data.mff.cuni.cz/soubory/číselníky/organizační-struktura.ofn.jsonld", format="json-ld")
print(list(g.query("""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n
SELECT DISTINCT ?rel ?com\n
        WHERE { \n
        ?rel a/rdfs:subPropertyOf* rdf:Property . \n
        OPTIONAL { ?rel rdfs:comment ?com } \n
        }
""")))

In [33]:
from shaclgen.shaclgen import data_graph
from rdflib import Graph

source_graph = Graph()
source_graph.parse("https://data.mff.cuni.cz/soubory/čoi/coi.trig", format="trig")

extraction_graph = data_graph(source_graph)
shacl_graph = extraction_graph.gen_graph()
print(shacl_graph)

2025-10-23 15:51:52.087 | INFO     | shaclgen.shaclgen:gen_graph:119 - Start Extraction of the Data Graph
2025-10-23 15:51:52.122 | INFO     | shaclgen.shaclgen:gen_graph:120 - Classes …
2025-10-23 15:51:52.409 | INFO     | shaclgen.shaclgen:gen_graph:122 - Properties …
2025-10-23 15:51:52.419 | INFO     | shaclgen.shaclgen:gen_graph:124 - Write resulting SHACL Graph …


[a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory']].


In [46]:
print(shacl_graph.serialize(format='trig'))
shacl_graph.print("trig")
